# **Mini Project : Pengambilan dan Pembersihan Data Melalui API** 

### Pengambilan dan Eksplorasi Data Lokasi Menggunakan Geoapify Places API



**Nama**  Fajrin Efantri

**Pelatihan**  AI Automation Engineer

**Project**  Pengambilan dan Pembersihan Data Melalui API


---

## **Bagian 1: Mendapatkan API Key**

Pada bagian ini, kita membutuhkan API Key untuk dapat mengakses data dari Geoapify API. API Key berfungsi sebagai kredensial untuk mengautentikasi setiap permintaan yang dikirimkan ke API Geoapify.

Setelah mendapatkan API Key, key tersebut disimpan dalam file **.env** agar tidak dituliskan secara langsung di dalam kode program.

### Langkah membuat API key di Geoapify

1. Buka halaman Geoapify MyProjects
2. Buat project baru
3. Masuk ke bagian API Keys
4. Salin API key yang dibuat



In [32]:
# Menginstall library yang dibutuhkan dalam mengakses API

!pip install requests python-dotenv --quiet

In [33]:
# Mengimport library yang dibutuhkan dalam mengaskes API
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("GEOAPIFY_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


---

## **Bagian 2: Mencoba Memanggil API**

Geoapify menyediakan API yang dapat digunakan untuk mencari informasi mengenai berbagai lokasi. Pada tahap ini, kita memanfaatkan endpoint Places untuk memperoleh data tempat berdasarkan kategori dan wilayah tertentu.

Agar pencarian dapat dilakukan sesuai kebutuhan, request API dilengkapi dengan beberapa parameter, yaitu:
1. categories → menentukan jenis tempat yang ingin dicari
2. filter → menentukan wilayah atau area pencarian berdasarkan koordinat
3. limit → membatasi jumlah data yang dikembalikan oleh API
4. apiKey → digunakan sebagai kunci autentikasi untuk mengakses layanan Geoapify

Dengan parameter tersebut, API akan mengembalikan data lokasi yang sesuai dengan kriteria pencarian.

In [34]:
alamat_api = "https://api.geoapify.com/v2/places"

# Parameter pencarian
parameter = {
    "categories": "catering.cafe",  # Mencari kedai kopi
    "filter": "circle:106.827153,-6.175392,3000", # Dalam radius 3000 meter dari koordinat Monas
    "limit": 5, # Minta 5 data dulu untuk uji coba
    "apiKey": API_KEY # Kartu identitas kita yang sudah dimuat sebelumnya
}

response = requests.get(alamat_api, params=parameter)
print(f"Status Code: {response.status_code}")

# Kalau berhasil (Status 200), kita ubah datanya jadi bentuk dictionary
if response.status_code == 200:
    hasil = response.json()
    
    # Geoapify mengembalikan data dalam format GeoJSON. 
    # Daftar tempatnya ada di dalam kunci "features".
    daftar_tempat = hasil.get("features", [])
    print(f"Jumlah tempat yang dikirim kali ini: {len(daftar_tempat)}")
else:
    print("Ada yang salah nih. Pesan dari server:", response.text)

Status Code: 200
Jumlah tempat yang dikirim kali ini: 5


In [35]:
# Menampilkan hasil data yang didapat setelah permintaan melalui API 
hasil['features']

[{'type': 'Feature',
  'properties': {'name': 'The Gade Coffee and Gold',
   'country': 'Indonesia',
   'country_code': 'id',
   'state': 'Java',
   'city': 'Central Jakarta',
   'postcode': '10110',
   'district': 'Gambir',
   'suburb': 'Gambir',
   'city_block': 'RW 02',
   'street': 'Jalan Haji Agus Salim',
   'iso3166_2': 'ID-JW',
   'iso3166_2_sublevel': 'ID-JK',
   'lon': 106.8247362,
   'lat': -6.1828398,
   'formatted': 'The Gade Coffee and Gold, Jalan Haji Agus Salim, Gambir, Central Jakarta 10110, Java, Indonesia',
   'address_line1': 'The Gade Coffee and Gold',
   'address_line2': 'Jalan Haji Agus Salim, Gambir, Central Jakarta 10110, Java, Indonesia',
   'categories': ['catering', 'catering.cafe'],
   'details': ['details'],
   'datasource': {'sourcename': 'openstreetmap',
    'attribution': '© OpenStreetMap contributors',
    'license': 'Open Database License',
    'url': 'https://www.openstreetmap.org/copyright',
    'raw': {'lat': -6.1828398,
     'lon': 106.8247362,
   

In [36]:
# Preview pada data yang telah diperoleh, kemudian menampilkan
# informasi dasar dari lokasi yang ditemukan

tempat = hasil["features"][0]["properties"]

print("Nama Tempat :", tempat["name"])
print("Alamat      :", tempat["formatted"])
print("Latitude    :", tempat["lat"])
print("Longitude   :", tempat["lon"])

Nama Tempat : The Gade Coffee and Gold
Alamat      : The Gade Coffee and Gold, Jalan Haji Agus Salim, Gambir, Central Jakarta 10110, Java, Indonesia
Latitude    : -6.1828398
Longitude   : 106.8247362


---

## **Bagian 3: Membungkus Jadi Class**

Pada tahap ini, proses pengambilan data dari Geoapify API akan dibuat dalam bentuk class agar kode lebih terstruktur dan dapat digunakan kembali.

Class `KlienTempat` digunakan untuk menyimpan API Key dan alamat endpoint, serta menjalankan proses pencarian lokasi berdasarkan kategori, area, dan jumlah data yang diinginkan.
Dengan menggunakan class, proses pemanggilan API menjadi lebih rapi dan tidak perlu menuliskan kembali kode yang sama setiap kali ingin melakukan pencarian data.

In [37]:
class KlienTempat:
    
    def __init__(self, api_key):
        # Menyimpan kartu identitas agar bisa dipakai oleh semua method di class ini
        self.api_key = api_key
        self.alamat_api = "https://api.geoapify.com/v2/places"
        
    def ambil_data(self, kategori, area_filter, jumlah=50):
        # Menyiapkan parameter pencarian
        parameter = {
            "categories": kategori,
            "filter": area_filter,
            "limit": jumlah,
            "apiKey": self.api_key
        }
        
        try:
            # Rencana utama: menghubungi Geoapify dengan batas waktu tunggu 20 detik
            response = requests.get(self.alamat_api, params=parameter, timeout=20)
        except Exception:
            # Rencana cadangan: kalau koneksi internet ngadat, tunggu 3 detik lalu coba lagi
            print("Koneksi bermasalah nih, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, timeout=20)
            
        # Cek apakah permintaan ditolak oleh server
        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame() # Mengembalikan tabel kosong kalau gagal
            
        # Ekstrak daftar tempat dari JSON
        daftar_tempat = response.json().get("features", [])
        
        data_bersih = []
        for tempat in daftar_tempat:
            info = tempat["properties"]
            # Memasukkan informasi yang sudah diekstrak ke dalam dictionary baru
            data_bersih.append({
                "Nama": info.get("name"),
                "Alamat Lengkap": info.get("formatted"),
                "Kategori Detail": ", ".join(info.get("categories", [])), # Jadikan string
                "Latitude": info.get("lat"),
                "Longitude": info.get("lon")
            })
            
        # Mengubah list of dictionary menjadi tabel (DataFrame)
        return pd.DataFrame(data_bersih)

print("Class KlienTempat siap dipakai!")

Class KlienTempat siap dipakai!


In [38]:
# Menampilkan Detail Data Lokasi

hasil["features"][0]["properties"]

{'name': 'The Gade Coffee and Gold',
 'country': 'Indonesia',
 'country_code': 'id',
 'state': 'Java',
 'city': 'Central Jakarta',
 'postcode': '10110',
 'district': 'Gambir',
 'suburb': 'Gambir',
 'city_block': 'RW 02',
 'street': 'Jalan Haji Agus Salim',
 'iso3166_2': 'ID-JW',
 'iso3166_2_sublevel': 'ID-JK',
 'lon': 106.8247362,
 'lat': -6.1828398,
 'formatted': 'The Gade Coffee and Gold, Jalan Haji Agus Salim, Gambir, Central Jakarta 10110, Java, Indonesia',
 'address_line1': 'The Gade Coffee and Gold',
 'address_line2': 'Jalan Haji Agus Salim, Gambir, Central Jakarta 10110, Java, Indonesia',
 'categories': ['catering', 'catering.cafe'],
 'details': ['details'],
 'datasource': {'sourcename': 'openstreetmap',
  'attribution': '© OpenStreetMap contributors',
  'license': 'Open Database License',
  'url': 'https://www.openstreetmap.org/copyright',
  'raw': {'lat': -6.1828398,
   'lon': 106.8247362,
   'name': 'The Gade Coffee and Gold',
   'branch': 'Kementrian BUMN',
   'osm_id': 9753

In [39]:
# Pastikan variabel API_KEY sudah jalan di tahap awal ya

# 1. Buat petugasnya 
klien = KlienTempat(API_KEY)

# 2. Siapkan skenario pencarian (contoh cari kafe, restoran, supermarket di radius Monas)
daftar_kategori = ["catering.cafe", "catering.restaurant", "commercial.supermarket"]
area_monas = "circle:106.827153,-6.175392,3000"

semua_tabel = []

for kategori in daftar_kategori:
    print(f"Mengambil data kategori: {kategori}...")
    # Memanggil method ambil_data dari class KlienTempat
    tabel = klien.ambil_data(kategori=kategori, area_filter=area_monas, jumlah=50)
    
    print(f"Dapat {len(tabel)} baris.")
    semua_tabel.append(tabel)
    
    # Jeda 1 detik supaya aman dari blokir server
    import time
    time.sleep(1)

# 3. Satukan semua tabel menjadi DataFrame df_tempat
df_tempat = pd.concat(semua_tabel, ignore_index=True)

print(f"\nTotal data terkumpul: {len(df_tempat)} baris")
df_tempat.head()

Mengambil data kategori: catering.cafe...
Dapat 50 baris.
Mengambil data kategori: catering.restaurant...
Dapat 50 baris.
Mengambil data kategori: commercial.supermarket...
Dapat 14 baris.

Total data terkumpul: 114 baris


,Nama,Alamat Lengkap,Kategori Detail,Latitude,Longitude
0,The Gade Coffee and Gold,"The Gade Coffee and Gold, Jalan Haji Agus Sali...","catering, catering.cafe",-6.182840,106.824736
1,Jakarta Coffee House,"Jakarta Coffee House, Jalan Kyai Haji Wahid Ha...","catering, catering.cafe",-6.187372,106.828101
2,Filosofi Kopi Pos Bloc,"Filosofi Kopi Pos Bloc, Jalan Pos, Sawah Besar...","catering, catering.cafe",-6.166904,106.833501
3,Kay's Cafe,"Kay's Cafe, Jalan Pintu Air Raya, Sawah Besar,...","catering, catering.cafe",-6.163003,106.832401
4,Excelso Express,"Excelso Express, Jalan Kenari II, Senen, Centr...","catering, catering.cafe",-6.194636,106.845977


In [40]:
# Melihat informasi data

df_tempat.info()

<class 'pandas.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Nama             113 non-null    str    
 1   Alamat Lengkap   114 non-null    str    
 2   Kategori Detail  114 non-null    str    
 3   Latitude         114 non-null    float64
 4   Longitude        114 non-null    float64
dtypes: float64(2), str(3)
memory usage: 21.4 KB


---

## **Bagian 4: Membersihkan Data (Data Cleaning)**

Pada tahap ini, data yang diperoleh dari Geoapify akan diperiksa dan dibersihkan agar dapat digunakan dengan lebih baik pada proses selanjutnya.
Pembersihan data dilakukan dengan memastikan informasi yang digunakan memiliki kondisi yang sesuai, seperti:
1. Nilai yang kosong → memeriksa data lokasi yang tidak memiliki informasi tertentu.
2. Data yang duplikat → memastikan tidak terdapat lokasi yang tercatat lebih dari satu kali.
3. Format atau tipe data → memastikan setiap informasi memiliki format dan tipe data yang sesuai untuk pengolahan.

In [41]:
# 1. Cek nilai kosong (NaN/None) di setiap kolom
print("1. Jumlah sel kosong per kolom:")
print(df_tempat.isnull().sum())
print()

# 2. Cek baris kembar
# Kita pakai patokan 'Alamat Lengkap' atau kombinasi 'Nama' dan 'Latitude' 
# karena kadang nama tempat bisa sama tapi cabangnya beda.
print("2. Jumlah baris yang kembar (berdasarkan Alamat Lengkap):")
print(df_tempat.duplicated(subset=["Alamat Lengkap"]).sum())
print()

# 3. Cek tipe data
# Pastikan kolom angka (Latitude, Longitude) tipenya beneran angka (float), bukan object/string
print("3. Tipe data setiap kolom:")
print(df_tempat.dtypes)

1. Jumlah sel kosong per kolom:
Nama               1
Alamat Lengkap     0
Kategori Detail    0
Latitude           0
Longitude          0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan Alamat Lengkap):
0

3. Tipe data setiap kolom:
Nama                   str
Alamat Lengkap         str
Kategori Detail        str
Latitude           float64
Longitude          float64
dtype: object


### **Membaca hasil pemeriksaan**

Berdasarkan pemeriksaan yang telah dilakukan, terdapat beberapa kondisi pada data yang perlu diperhatikan sebelum masuk ke tahap pengolahan berikutnya.

Pertama, terdapat 1 data kosong pada kolom `Nama`. Sementara itu, kolom `Alamat Lengkap`, `Kategori Detail`, `Latitude`, dan `Longitude` tidak memiliki data yang kosong. Data kosong pada kolom `Nama` akan ditangani agar tidak mengganggu proses pengolahan data selanjutnya.

Kedua, berdasarkan pemeriksaan menggunakan kolom `Alamat Lengkap`, tidak ditemukan adanya baris data yang duplikat. Hal ini menunjukkan bahwa setiap lokasi dalam data memiliki alamat yang berbeda.

Ketiga, berdasarkan pemeriksaan tipe data, seluruh kolom telah memiliki tipe data yang sesuai. Kolom `Nama`, `Alamat Lengkap`, dan `Kategori` Detail bertipe `str`, sedangkan `Latitude` dan `Longitude` bertipe `float64`. Dengan demikian, tidak diperlukan perubahan tipe data untuk kolom-kolom tersebut.

### Menangani Nilai Kosong

In [42]:
def bersihkan_nama(teks):
    if pd.isna(teks) or teks == "":
        return "Nama Tidak Diketahui"
    return teks

df_tempat["Nama"] = df_tempat["Nama"].apply(bersihkan_nama)

print("Jumlah nilai kosong pada kolom Nama setelah dibersihkan:")
print(df_tempat["Nama"].isnull().sum())

Jumlah nilai kosong pada kolom Nama setelah dibersihkan:
0


### Menangani Baris Kembar

In [43]:
jumlah_sebelum = len(df_tempat)

df_bersih = df_tempat.drop_duplicates(subset="Alamat Lengkap").copy()

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")

Jumlah baris sebelum : 114
Jumlah baris sesudah : 114
Baris kembar dibuang : 0


In [44]:
import pandas as pd
import numpy as np

# --- FUNGSI PEMBERSIH ---
# (Syarat wajib: minimal 2 function pembersih)

# Function 1: Menangani nilai kosong pada kolom teks
def bersihkan_nama(teks):
    # Menggunakan pd.isna() karena lebih aman untuk mendeteksi NaN/None dari pandas
    if pd.isna(teks) or teks == "":
        return "Nama Tidak Diketahui"
    return teks

# Function 2: Memastikan kolom koordinat bertipe angka (float)
def pastikan_tipe_angka(nilai):
    # Memaksa ubah ke float kalau kebetulan ada yang nyelip berbentuk string
    return float(nilai)

# --- PROSES EKSEKUSI PEMBERSIHAN ---

# 1. Tangani nilai kosong
df_tempat["Nama"] = df_tempat["Nama"].apply(bersihkan_nama)

# 2. Tangani baris kembar (meskipun 0, kode ini memastikan tidak ada kebocoran duplikat)
jumlah_sebelum = len(df_tempat)
df_bersih = df_tempat.drop_duplicates(subset=["Alamat Lengkap"]).copy()
jumlah_sesudah = len(df_bersih)
print(f"Baris kembar dibuang : {jumlah_sebelum - jumlah_sesudah}")

# 3. Tangani tipe data
df_bersih["Latitude"] = df_bersih["Latitude"].apply(pastikan_tipe_angka)
df_bersih["Longitude"] = df_bersih["Longitude"].apply(pastikan_tipe_angka)

# --- CEK HASIL AKHIR ---
print("\n--- Pengecekan Setelah Dibersihkan ---")
print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Nama masih ada kosong  : {df_bersih['Nama'].isnull().sum()}")

Baris kembar dibuang : 0

--- Pengecekan Setelah Dibersihkan ---
Jumlah baris           : 114
Sudah lebih dari 100?  : True
Nama masih ada kosong  : 0


### Menangani Tipe Data


In [45]:
def pastikan_tipe_angka(nilai):
    return pd.to_numeric(nilai, errors="coerce")


print("Tipe data sebelum:")
print(df_bersih[["Latitude", "Longitude"]].dtypes)

df_bersih["Latitude"] = df_bersih["Latitude"].apply(pastikan_tipe_angka)
df_bersih["Longitude"] = df_bersih["Longitude"].apply(pastikan_tipe_angka)

print("\nTipe data sesudah:")
print(df_bersih[["Latitude", "Longitude"]].dtypes)

df_bersih.head()

Tipe data sebelum:
Latitude     float64
Longitude    float64
dtype: object

Tipe data sesudah:
Latitude     float64
Longitude    float64
dtype: object


,Nama,Alamat Lengkap,Kategori Detail,Latitude,Longitude
0,The Gade Coffee and Gold,"The Gade Coffee and Gold, Jalan Haji Agus Sali...","catering, catering.cafe",-6.182840,106.824736
1,Jakarta Coffee House,"Jakarta Coffee House, Jalan Kyai Haji Wahid Ha...","catering, catering.cafe",-6.187372,106.828101
2,Filosofi Kopi Pos Bloc,"Filosofi Kopi Pos Bloc, Jalan Pos, Sawah Besar...","catering, catering.cafe",-6.166904,106.833501
3,Kay's Cafe,"Kay's Cafe, Jalan Pintu Air Raya, Sawah Besar,...","catering, catering.cafe",-6.163003,106.832401
4,Excelso Express,"Excelso Express, Jalan Kenari II, Senen, Centr...","catering, catering.cafe",-6.194636,106.845977


In [46]:
# Pemeriksaan terakhir sebelum dianggap selesai

df_bersih.info()

<class 'pandas.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Nama             114 non-null    str    
 1   Alamat Lengkap   114 non-null    str    
 2   Kategori Detail  114 non-null    str    
 3   Latitude         114 non-null    float64
 4   Longitude        114 non-null    float64
dtypes: float64(2), str(3)
memory usage: 21.4 KB


## **Bagian 5: Menyimpan Hasil**

Menyimpan dataframe hasil cleaning ke file CSV

In [47]:
# Menyimpan dataframe hasil cleaning ke file CSV
df_bersih.to_csv("dataset_tempat_geoapify.csv", index=False)

print("Data berhasil disimpan ke file: dataset_tempat_geoapify.csv")
print(f"File berisi {len(df_bersih)} baris dan {len(df_bersih.columns)} kolom")

# Preview data yang telah disimpan
df_bersih.head()

Data berhasil disimpan ke file: dataset_tempat_geoapify.csv
File berisi 114 baris dan 5 kolom


,Nama,Alamat Lengkap,Kategori Detail,Latitude,Longitude
0,The Gade Coffee and Gold,"The Gade Coffee and Gold, Jalan Haji Agus Sali...","catering, catering.cafe",-6.182840,106.824736
1,Jakarta Coffee House,"Jakarta Coffee House, Jalan Kyai Haji Wahid Ha...","catering, catering.cafe",-6.187372,106.828101
2,Filosofi Kopi Pos Bloc,"Filosofi Kopi Pos Bloc, Jalan Pos, Sawah Besar...","catering, catering.cafe",-6.166904,106.833501
3,Kay's Cafe,"Kay's Cafe, Jalan Pintu Air Raya, Sawah Besar,...","catering, catering.cafe",-6.163003,106.832401
4,Excelso Express,"Excelso Express, Jalan Kenari II, Senen, Centr...","catering, catering.cafe",-6.194636,106.845977


## **Bagian 6: Bahan untuk Slide**

In [48]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)
print("Sumber data      : Geoapify Places API")
print(f"Kategori dipakai : {', '.join(daftar_kategori)}")
print()
# Menampilkan perbandingan jumlah baris sebelum dan sesudah dibersihkan
print(f"Baris sebelum dibersihkan : {len(df_tempat)}")
print(f"Baris dataset akhir       : {len(df_bersih)}")
print()
print("Class yang dibuat:")
print("  1. KlienTempat - mengambil data tempat dari Geoapify")
print()
print("Function yang dibuat:")
print("  1. bersihkan_nama      - mengisi nama kosong dengan penanda")
print("  2. pastikan_tipe_angka - memastikan tipe data koordinat berbentuk desimal (float)")
print()
print("Temuan dari pembersihan data:")
print(f"  Baris kembar dibuang        : {len(df_tempat) - len(df_bersih)}")
print("  Nama kosong diberi penanda  : 1")
print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data      : Geoapify Places API
Kategori dipakai : catering.cafe, catering.restaurant, commercial.supermarket

Baris sebelum dibersihkan : 114
Baris dataset akhir       : 114

Class yang dibuat:
  1. KlienTempat - mengambil data tempat dari Geoapify

Function yang dibuat:
  1. bersihkan_nama      - mengisi nama kosong dengan penanda
  2. pastikan_tipe_angka - memastikan tipe data koordinat berbentuk desimal (float)

Temuan dari pembersihan data:
  Baris kembar dibuang        : 0
  Nama kosong diberi penanda  : 1
